# 2.2. Chuẩn hóa LaTeX/Toán học và Khử trùng 2 Tầng Xuyên Phiên bản

## 1. Quy tắc làm sạch ngữ nghĩa LaTeX & đồng nhất biểu thức toán (`normalize.py`)

- **Gỡ wrapper hình thức & lệnh dàn trang rỗng nghĩa**: Bóc đệ quy các lệnh trình bày (`\textbf`, `\emph`, `\caption`, `\footnote`...) giữ lại nội dung bên trong; chuyển `\href{url}{text}` thành `text (url)`; loại bỏ các lệnh bố cục (`\centering`, `\toprule`, `\midrule`, `\bottomrule`, `\vspace`, `\label`...) và chuẩn hóa Unicode `NFC`.
- **Đồng nhất biểu thức toán học 2 pha**: Bảo vệ khoảng trắng trong `\text{...}`/`\mbox{...}`, co khoảng trắng quanh toán tử để tăng tỷ lệ khớp băm, chuẩn hóa toàn bộ inline math (`$...$`, `\(...\)`) về `$...$` và toàn bộ block math (`$$...$$`, `\[...\]`, `equation*`, `align`, `gather`, `multline`...) về `\begin{equation}...\end{equation}`.

## 2. Khử trùng 2 tầng xuyên phiên bản (`bibliography.py` & `deduplicate.py`)

- **Tầng 1 — Hợp nhất tài liệu tham khảo & viết lại `\cite{}`**: Ghép các mục từ `.bib`, `\bibitem` và fallback `.bbl` theo định danh mạnh (`doi`, `eprint` arXiv) hoặc chữ ký `(title, authors)`, hợp nhất trường (*field unionization*) từ mọi phiên bản và **đổi tên mọi lệnh `\cite{}`** (kể cả bên trong công thức toán) về khóa đại diện duy nhất.
- **Tầng 2 — Kho phần tử `elements` dùng chung theo SHA-256**: Mỗi nút cây sau khi chuẩn hóa và đổi khóa `\cite{}` được băm theo `(arxiv_id, kind, normalized_content)` để lưu duy nhất một lần trong `elements`, trong khi bảng `hierarchy` lưu riêng cấu trúc cạnh `(version, element_id, parent_id)` của từng phiên bản.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
from parser.normalize import normalize
from parser.bibliography import parse_bibitems, parse_bibtex, unify_entries
from parser.hierarchy import parse_hierarchy
from parser.deduplicate import deduplicate

norm_samples = [
    ("Wrapper ngữ nghĩa & \\href + \\cite", r"We follow \textbf{Vaswani et al.} \href{https://arxiv.org/abs/1706.03762}{\emph{Transformer}} \cite{vit_transformer}.", "sentence"),
    ("Bảng & xóa lệnh dàn trang rỗng nghĩa", r"\centering\begin{tabular}{cc}\toprule A & B \\ \midrule 1 & 2 \\ \bottomrule\end{tabular}", "figure"),
    ("Đồng nhất toán inline & co toán tử", r"Given \( a + b = c \) and $x_1 + x_2 = y$.", "sentence"),
    ("Đồng nhất toán khối, bảo vệ \\text{} & đổi \\cite{}", r"E = m c^2 + p c \quad \text{in Eq.} ~ \cite{vit_transformer}", "block_formula"),
]
df_norm = pd.DataFrame([
    {
        "Trường hợp kiểm thử": name,
        "Đầu vào LaTeX thô": raw,
        "Sau chuẩn hóa & ánh xạ khóa": normalize(raw, {"vit_transformer": "vaswani2017"}, kind=k),
    }
    for name, raw, k in norm_samples
])
display(style_table(df_norm, caption="Bảng 7.1. Đối chiếu trực tiếp đầu vào LaTeX thô và đầu ra sau chuẩn hóa (`normalize.py`)."))

# Kịch bản 3 phiên bản chuẩn theo Đề bài M2: hợp nhất trường BibTeX + đổi \cite{} + dùng chung kho elements
v1_tex = r"""\documentclass{article}\title{Citation Demo v1}\begin{document}
\section{Introduction}
We study \(a + b = c\) following \textbf{Vaswani et al.} \cite{vaswani2017}.
\begin{thebibliography}{9}
\bibitem{vaswani2017} A. Vaswani and N. Shazeer.\newblock Attention Is All You Need.\newblock 2017.
\end{thebibliography}\end{document}"""

v2_tex = r"""\documentclass{article}\title{Citation Demo v2}\begin{document}
\section{Background and Model}
We study $a+b=c$ following {Vaswani et al.} \cite{vit_transformer}.
\begin{align*}E = m c^2 + p c \quad \text{in Eq.} ~ \cite{vit_transformer}\end{align*}
\end{document}"""
v2_bib = r"""@article{vit_transformer,
  author = {A. Vaswani and N. Shazeer},
  title = {Attention Is All You Need},
  year = {2017},
  doi = {10.5555/3295222.3295349}
}"""

v3_tex = r"""\documentclass{article}\title{Citation Demo v3}\begin{document}
\section{Background and Model}
We study $a + b = c$ following \emph{Vaswani et al.} \cite{vit_transformer}.
\[ E=m c^2+p c \quad \text{in Eq.} ~ \cite{vit_transformer} \]
\end{document}"""
v3_bib = r"""@inproceedings{vit_transformer,
  author = {Vaswani, A. and Shazeer, N.},
  title = {Attention Is All You Need},
  journal = {NeurIPS},
  url = {https://arxiv.org/abs/1706.03762}
}"""

h1, h2, h3 = parse_hierarchy(v1_tex), parse_hierarchy(v2_tex), parse_hierarchy(v3_tex)
raw_entries = [*parse_bibitems(v1_tex, 1), *parse_bibtex(v2_bib, 2), *parse_bibtex(v3_bib, 3)]
bib_rows, mappings, conflicts = unify_entries("2310.12345", raw_entries)
els, edges, _ = deduplicate("2310.12345", {1: h1, 2: h2, 3: h3}, mappings)

print("Ánh xạ citation_key xuyên 3 phiên bản:", mappings)
display(style_table(
    pd.DataFrame(bib_rows).drop(columns=["arxiv_id"]),
    caption="Bảng 7.2. Bản ghi `bibtex` duy nhất sau hợp nhất (union trọn vẹn `title, authors, year, doi, journal, url` từ cả 3 phiên bản).",
))

short_id = {e["element_id"]: e["element_id"][:21] + "…" for e in els}
display(style_table(
    pd.DataFrame([{"element_id (rút gọn)": short_id[e["element_id"]], "kind": e["kind"], "content": e["content"]} for e in els]),
    caption=f"Bảng 7.3. Kho `elements` dùng chung xuyên 3 phiên bản ({len(h1.nodes)+len(h2.nodes)+len(h3.nodes)} nút thô → chỉ còn {len(els)} phần tử duy nhất).",
))

kind_by_id = {e["element_id"]: e["kind"] for e in els}
display(style_table(
    pd.DataFrame([
        {
            "version": ed["version"],
            "child_kind": kind_by_id[ed["element_id"]],
            "element_id": short_id[ed["element_id"]],
            "parent_kind": kind_by_id[ed["parent_id"]],
            "parent_id": short_id[ed["parent_id"]],
        }
        for ed in edges
    ]),
    caption="Bảng 7.4. Cạnh `hierarchy` theo từng phiên bản: cùng 1 phần tử `sentence` gắn vào `Introduction` ở v1 nhưng gắn vào `Background and Model` ở v2 & v3.",
    int_cols=["version"],
))

## 3. Phân tích định lượng trên 25.000 bài: Sự khác biệt giữa Bài 1-Version và Bài Đa-Version

In [ ]:
rg = m2["dedup_by_revision_group"]
df_rg = pd.DataFrame([
    {
        "Nhóm bài báo": "Bài chỉ có 1 phiên bản (v1 duy nhất)",
        "Số bài báo": rg["single_version"]["publications"],
        "Elements trước": rg["single_version"]["elements_before"],
        "Elements sau": rg["single_version"]["elements_after"],
        "Giảm Elements (%)": 100.0 * rg["single_version"]["elements_reduction_rate"],
        "BibTeX trước": rg["single_version"]["bibtex_before"],
        "BibTeX sau": rg["single_version"]["bibtex_after"],
        "Giảm BibTeX (%)": 100.0 * rg["single_version"]["bibtex_reduction_rate"],
    },
    {
        "Nhóm bài báo": "Bài có ≥ 2 phiên bản (đa revision v1..vN)",
        "Số bài báo": rg["multi_version"]["publications"],
        "Elements trước": rg["multi_version"]["elements_before"],
        "Elements sau": rg["multi_version"]["elements_after"],
        "Giảm Elements (%)": 100.0 * rg["multi_version"]["elements_reduction_rate"],
        "BibTeX trước": rg["multi_version"]["bibtex_before"],
        "BibTeX sau": rg["multi_version"]["bibtex_after"],
        "Giảm BibTeX (%)": 100.0 * rg["multi_version"]["bibtex_reduction_rate"],
    },
    {
        "Nhóm bài báo": "Toàn tập hợp nhất (25.000 bài báo)",
        "Số bài báo": scope["papers"],
        "Elements trước": m2["deduplication"]["elements"]["before"],
        "Elements sau": m2["deduplication"]["elements"]["after"],
        "Giảm Elements (%)": 100.0 * m2["deduplication"]["elements"]["reduction_rate"],
        "BibTeX trước": m2["deduplication"]["bibtex"]["before"],
        "BibTeX sau": m2["deduplication"]["bibtex"]["after"],
        "Giảm BibTeX (%)": 100.0 * m2["deduplication"]["bibtex"]["reduction_rate"],
    },
])
display(style_table(
    df_rg,
    caption="Bảng 7.5. Giải phẫu tỷ lệ khử trùng: trên 10.933 bài đa phiên bản, khử trùng cắt giảm tới 46,95% Elements và 50,41% BibTeX!",
    int_cols=["Số bài báo", "Elements trước", "Elements sau", "BibTeX trước", "BibTeX sau"],
    pct_cols=["Giảm Elements (%)", "Giảm BibTeX (%)"],
    bar_cols={"Giảm BibTeX (%)": ("rgba(13, 148, 136, 0.22)", 60.0)},
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.0))
xr = np.arange(3); wr = 0.34
ax1.bar(xr - wr/2, df_rg["Giảm Elements (%)"], wr, color=BOOK_PALETTE["navy"], label="Giảm Elements (%)")
ax1.bar(xr + wr/2, df_rg["Giảm BibTeX (%)"], wr, color=BOOK_PALETTE["teal"], label="Giảm BibTeX (%)")
ax1.set_xticks(xr); ax1.set_xticklabels(["1 phiên bản\n(14.067 bài)", "Đa phiên bản ≥2\n(10.933 bài)", "Toàn tập\n(25.000 bài)"])
ax1.set_ylim(0, 60); ax1.grid(axis="y"); ax1.legend(fontsize=8.3)
ax1.set_ylabel("Tỷ lệ bản sao loại bỏ (%)")
ax1.set_title("Hình 7.1a. Đột phá khử trùng trên nhóm bài đa phiên bản")
for idx_g, r_g in df_rg.iterrows():
    ax1.text(idx_g - wr/2, r_g["Giảm Elements (%)"] + 1.1, fmt_pct(r_g["Giảm Elements (%)"], 1), ha="center", fontsize=8.0, fontweight="bold")
    ax1.text(idx_g + wr/2, r_g["Giảm BibTeX (%)"] + 1.1, fmt_pct(r_g["Giảm BibTeX (%)"], 1), ha="center", fontsize=8.0, fontweight="bold", color=BOOK_PALETTE["teal"])

if "bibtex_profile" in m2:
    bp = m2["bibtex_profile"]["field_nonnull_rates"]
    s_bp = pd.Series({k: v * 100.0 for k, v in bp.items()}).sort_values(ascending=True)
    ax2.barh(s_bp.index, s_bp.values, color=BOOK_PALETTE["sky"], height=0.55)
    ax2.set_xlim(0, 115); ax2.grid(axis="x")
    ax2.set_xlabel("Tỷ lệ có dữ liệu (%)")
    ax2.set_title("Hình 7.1b. Độ phủ trường trên 1.177.293 mục BibTeX")
    for i_b, v_b in enumerate(s_bp.values):
        ax2.text(v_b + 1.5, i_b, fmt_pct(v_b, 1), va="center", fontsize=8.2, fontweight="bold")
plt.tight_layout(); plt.show()

```{note}
**Minh bạch kiểm toán các sự kiện biên trong khử trùng:**
Trên toàn tập 25.000 bài có `1.206` phiên bản (`1.026` `error` + `180` `error_missing_source`) được đánh dấu `versions_with_unmeasured_counts` để không làm sai lệch mẫu số trước/sau khử trùng; đồng thời hệ thống ghi nhận `75.595` khóa trích dẫn chưa khớp mục tài liệu tham khảo và `69.929` sự kiện xung đột giá trị trường giữa các phiên bản (luôn bảo toàn giá trị xuất hiện đầu tiên kèm nhật ký kiểm toán).
```

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/normalize.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/normalize.py).
- Mã nguồn triển khai: [`src/parser/bibliography.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/bibliography.py).
- Mã nguồn triển khai: [`src/parser/deduplicate.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/deduplicate.py).
- Mã nguồn triển khai: [`src/parser/statistics.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/statistics.py).